# DEXTRA · 09 — Paper figures and tables

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/09_paper_assets.ipynb)

Notebook 08 finished all fifteen folds and wrote its analysis to the dataset as CSVs.
This notebook turns those CSVs into the figures and LaTeX tables a paper needs.

**It trains nothing.** No GPU, no HuggingFace token, no secrets — the dataset is public.
It runs in well under a minute and can be re-run as often as you like.

At the end it offers `dextra_paper_assets.zip` for download: every figure as 300-dpi PNG
**and** PDF (use the PDF in LaTeX), plus `.tex` tables you can paste straight into an
IEEEtran document.

In [ ]:
# ── setup: only matplotlib and pandas are needed ────────────────────────────
import importlib, subprocess, sys

for mod, spec in [("pandas", "pandas"), ("matplotlib", "matplotlib"), ("numpy", "numpy")]:
    try:
        importlib.import_module(mod)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)

import io
import json
import os
import re
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if not IN_COLAB:
    matplotlib.use("Agg")

REPO = "Aduitya9/Aduitya_dextra-aqi-india"
BASE = f"https://huggingface.co/datasets/{REPO}/resolve/main/processed"
OUT = Path("paper_assets")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.autolayout": True,
})
# colour-blind safe
C_FINAL, C_BASE, C_REF, C_GREY = "#0072B2", "#D55E00", "#009E73", "#999999"
print(f"pandas {pd.__version__} | matplotlib {matplotlib.__version__} | colab={IN_COLAB}")

In [ ]:
# ── fetch notebook 08's analysis ────────────────────────────────────────────
WANTED = ["final_choice.json", "final_selection.csv", "final_scores.csv",
          "final_verdict.csv", "final_per_city.csv", "final_noise_ceiling.csv",
          "final_ablation_next_day.csv", "final_ablation_all.csv",
          "final_alarm_next_day.csv", "final_anchor_weights.csv"]
D = {}
for name in WANTED:
    try:
        with urllib.request.urlopen(f"{BASE}/{name}", timeout=60) as r:
            raw = r.read()
        D[name] = json.loads(raw) if name.endswith(".json") else pd.read_csv(io.BytesIO(raw))
        n = "ok" if name.endswith(".json") else f"{D[name].shape[0]}x{D[name].shape[1]}"
        print(f"  loaded  {name:34s} {n}")
    except Exception as e:
        print(f"  MISSING {name:34s} ({type(e).__name__}) - figures using it are skipped")

assert "final_scores.csv" in D, "final_scores.csv is required; did notebook 08 finish?"
choice = D.get("final_choice.json", {})
FINAL_NAME = choice.get("final_model", "dextra")
print(f"\nfinal model selected by notebook 08: {FINAL_NAME!r} "
      f"({choice.get('folds','?')} of {choice.get('of','?')} folds, seeds {choice.get('seeds')})")

# the row in final_scores that corresponds to the selected model
FINAL_ROW = "+ anchored forecast and 72-hour tokens"
PERSIST_ROW = "Persistence"


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUT / "figures" / f"{name}.{ext}", bbox_inches="tight")
    print(f"  wrote figures/{name}.png + .pdf")
    if IN_COLAB:
        try:
            from IPython.display import display
            display(fig)
        except Exception:
            pass
    plt.close(fig)

## Figure 1 — the noise ceiling

This is the strongest single result in the project. The ceiling is the error that no
forecaster could remove, estimated from repeated measurement of the same city-days. Each
bar is the share of the *reducible* error a forecaster actually removes, so the axis has
a meaningful 0 (persistence) and a meaningful 1 (the ceiling).

In [ ]:
if "final_noise_ceiling.csv" in D:
    nc = D["final_noise_ceiling.csv"].copy()
    col = "share of reducible error removed"
    nc = nc[~nc["forecaster"].str.contains("noise ceiling", case=False)]
    nc = nc.sort_values(col)
    fig, ax = plt.subplots(figsize=(7.2, 5.4))
    colours = [C_FINAL if f == FINAL_ROW else (C_REF if f.startswith("+") else C_BASE)
               for f in nc["forecaster"]]
    ax.barh(range(len(nc)), nc[col] * 100, color=colours, alpha=0.9)
    ax.set_yticks(range(len(nc)))
    ax.set_yticklabels(nc["forecaster"], fontsize=7.5)
    ax.set_xlabel("share of reducible error removed (%)")
    ax.set_xlim(0, 100)
    ax.axvline(100, color="k", lw=1.2)
    ax.text(99, -0.9, "noise ceiling", ha="right", fontsize=7.5, style="italic")
    for i, (v, f) in enumerate(zip(nc[col], nc["forecaster"])):
        ax.text(v * 100 + 1, i, f"{v*100:.1f}", va="center", fontsize=7,
                fontweight="bold" if f == FINAL_ROW else "normal")
    ax.set_title("How much of the removable error each forecaster actually removes\n"
                 "(next-day; 0 = persistence, 100 = the irreducible noise floor)", fontsize=9)
    save_fig(fig, "fig1_noise_ceiling")

    ceil = D["final_noise_ceiling.csv"]
    c_rmse = float(ceil.loc[ceil["forecaster"].str.contains("ceiling", case=False),
                            "next-day RMSE"].iloc[0])
    p_rmse = float(ceil.loc[ceil["forecaster"] == PERSIST_ROW, "next-day RMSE"].iloc[0])
    f_rmse = float(ceil.loc[ceil["forecaster"] == FINAL_ROW, "next-day RMSE"].iloc[0])
    # Share removed is measured on SQUARED error, matching the column plotted above and
    # the dataset's own definition. Computing it on RMSE instead gives a different (and
    # inconsistent) number, so do not mix the two.
    share = (p_rmse ** 2 - f_rmse ** 2) / (p_rmse ** 2 - c_rmse ** 2) * 100
    print(f"\n  noise floor      {c_rmse:6.2f} AQI RMSE")
    print(f"  persistence      {p_rmse:6.2f}")
    print(f"  final model      {f_rmse:6.2f}   -> removes {share:.1f}% of the reducible "
          f"error (squared-error basis)")

## Figure 2 — what the final model beats, and what it does not

A forest plot of the pre-registered comparisons. A bar whose interval crosses zero is
**not** a win: report those as ties, not as victories.

One subtlety to explain if a reviewer asks: the **intervals are unadjusted** bootstrap
CIs, while the **colour uses the Holm-adjusted p-value**. Holm is stricter, so a couple
of points sit just clear of zero and are still counted as ties. That is deliberate — the
pre-registration committed to Holm correction across the comparison family, and the
stricter answer is the one that counts.

In [ ]:
def parse_ci(s):
    m = re.findall(r"[-+]?\d*\.?\d+", str(s))
    return (float(m[0]), float(m[1])) if len(m) >= 2 else (np.nan, np.nan)


if "final_verdict.csv" in D:
    v = D["final_verdict.csv"].copy()
    v = v[~v["final against"].str.startswith("+")]      # keep rivals, drop ablation rungs
    v[["lo", "hi"]] = v["AP 95%"].apply(lambda s: pd.Series(parse_ci(s)))
    v = v.sort_values("AP difference")
    sig = v["AP p (Holm)"] < 0.05

    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    y = np.arange(len(v))
    ax.errorbar(v["AP difference"], y,
                xerr=[v["AP difference"] - v["lo"], v["hi"] - v["AP difference"]],
                fmt="o", ms=5, lw=1.4, capsize=3,
                color=C_GREY, ecolor=C_GREY, zorder=2)
    ax.scatter(v["AP difference"][sig.values], y[sig.values], s=55, color=C_FINAL, zorder=3,
               label="significant after Holm (p < 0.05)")
    ax.scatter(v["AP difference"][~sig.values], y[~sig.values], s=55, color=C_BASE,
               zorder=3, label="not significant - a tie")
    ax.axvline(0, color="k", lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels(v["final against"], fontsize=7.5)
    ax.set_xlabel("average-precision gain of the final model (95% CI)")
    ax.set_title("Severe-day detection: the final model against each rival", fontsize=9)
    ax.legend(fontsize=7, loc="lower right")
    save_fig(fig, "fig2_verdict_forest")

    print("\n  significant wins :", " | ".join(v.loc[sig.values, "final against"]))
    print("  ties (report as ties, not wins):",
          " | ".join(v.loc[~sig.values, "final against"]))

## Figure 3 — the ablation ladder

One change per rung, so a gain can be traced to its cause. RMSE falls steadily; average
precision barely moves, which is itself the finding.

In [ ]:
if "final_scores.csv" in D:
    s = D["final_scores.csv"].copy()
    rungs = [r for r in s["forecaster"] if r.startswith("+")]
    ladder = ["All-token head"] + rungs
    lad = s[s["forecaster"].isin(ladder)].set_index("forecaster").reindex(ladder).reset_index()
    short = [r.replace("+ ", "").replace(" (control)", "")[:28] for r in lad["forecaster"]]

    fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.8))
    for ax, col, lab, better in (
            (axes[0], "next-day RMSE", "next-day RMSE (AQI points)", "lower is better"),
            (axes[1], "next-day AP", "next-day average precision", "higher is better")):
        ax.plot(range(len(lad)), lad[col], marker="o", ms=5, lw=1.8, color=C_FINAL)
        ax.set_xticks(range(len(lad)))
        ax.set_xticklabels(short, rotation=35, ha="right", fontsize=7)
        ax.set_ylabel(lab)
        ax.set_title(f"{lab} — {better}", fontsize=9)
    save_fig(fig, "fig3_ablation_ladder")

## Figure 4 — per city

Severe-day counts differ by more than an order of magnitude across cities, so a pooled
number hides a lot. This figure is the honest version.

In [ ]:
if "final_per_city.csv" in D:
    pc = D["final_per_city.csv"].copy().sort_values("severe", ascending=False)
    cols = [("AP persistence", "Persistence", C_GREY),
            ("AP gbm hourly", "GBM + hourly", C_REF),
            ("AP chronos-2", "Chronos-2", C_BASE),
            ("AP final", "Final model", C_FINAL)]
    cols = [c for c in cols if c[0] in pc.columns]
    fig, ax = plt.subplots(figsize=(8.6, 3.9))
    w = 0.8 / len(cols)
    x = np.arange(len(pc))
    for i, (col, lab, colr) in enumerate(cols):
        ax.bar(x + i * w - 0.4 + w / 2, pc[col], width=w, label=lab, color=colr, alpha=0.92)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}\n({n} severe)" for c, n in zip(pc["city"], pc["severe"])],
                       fontsize=7.5)
    ax.set_ylabel("average precision")
    ax.set_title("Severe-day detection per held-out city", fontsize=9)
    ax.legend(fontsize=7.5, ncol=4)
    save_fig(fig, "fig4_per_city")
    print("\n  Note the spread: pooled AP is dominated by the cities with many severe days.")

## Figure 5 — episode onsets caught

The operationally useful number: an advisory issued on the *first* bad day is worth far
more than one issued on the third.

In [ ]:
if "final_scores.csv" in D and "onsets caught" in D["final_scores.csv"].columns:
    s = D["final_scores.csv"].copy().sort_values("onsets caught")
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    colours = [C_FINAL if f == FINAL_ROW else (C_REF if f.startswith("+") else C_BASE)
               for f in s["forecaster"]]
    ax.barh(range(len(s)), s["onsets caught"], color=colours, alpha=0.9)
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels(s["forecaster"], fontsize=7.5)
    ax.set_xlabel("episode onsets caught (mean over seeds)")
    ax.set_title("How many episode onsets each forecaster catches", fontsize=9)
    for i, v in enumerate(s["onsets caught"]):
        ax.text(v + 1, i, f"{v:.0f}", va="center", fontsize=7)
    save_fig(fig, "fig5_onsets")

## LaTeX tables

Written to `paper_assets/tables/*.tex`. They use `booktabs`, so put
`\usepackage{booktabs}` in your preamble.

In [ ]:
def tex_escape(s):
    s = str(s)
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
                 ("#", r"\#"), ("_", r"\_"), ("{", r"\{"), ("}", r"\}"),
                 ("~", r"\textasciitilde{}"), ("^", r"\textasciicircum{}")]:
        s = s.replace(a, b)
    return s


def to_tex(df, name, caption, label, floatfmt="%.3f", bold_rows=()):
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_numeric_dtype(d[c]):
            d[c] = d[c].map(lambda v: "" if pd.isna(v) else floatfmt % v)
    head = " & ".join(tex_escape(c) for c in d.columns) + r" \\"
    lines = []
    for _, r in d.iterrows():
        cells = [tex_escape(v) for v in r]
        if str(r.iloc[0]) in bold_rows:
            cells = [r"\textbf{" + c + "}" for c in cells]
        lines.append(" & ".join(cells) + r" \\")
    tex = "\n".join([
        r"\begin{table}[t]", r"\centering",
        r"\caption{" + caption + "}", r"\label{" + label + "}",
        r"\begin{tabular}{l" + "r" * (len(d.columns) - 1) + "}", r"\toprule",
        head, r"\midrule", *lines, r"\bottomrule", r"\end{tabular}", r"\end{table}", ""])
    (OUT / "tables" / f"{name}.tex").write_text(tex, encoding="utf-8")
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
    print(f"  wrote tables/{name}.tex (+ .csv)")


if "final_selection.csv" in D:
    to_tex(D["final_selection.csv"], "table1_selection",
           "Candidate selection on validation rows only. The pre-registered rule picks the "
           "lowest validation RMSE among candidates within 0.002 average precision of the "
           "control.", "tab:selection", bold_rows=(FINAL_NAME,))

if "final_scores.csv" in D:
    keep = [c for c in ["forecaster", "AP", "onset AP", "next-day AP", "RMSE",
                        "next-day RMSE", "onsets caught"] if c in D["final_scores.csv"].columns]
    to_tex(D["final_scores.csv"][keep], "table2_scores",
           "Leave-one-city-out results across all fifteen cities, three seeds.",
           "tab:scores", bold_rows=(FINAL_ROW,))

if "final_verdict.csv" in D:
    v = D["final_verdict.csv"]
    keep = [c for c in ["final against", "AP difference", "AP 95%", "AP p (Holm)",
                        "RMSE difference", "RMSE p (Holm)"] if c in v.columns]
    to_tex(v[keep], "table3_verdict",
           "The final model against each rival, with Holm-corrected p-values. Intervals "
           "spanning zero are reported as ties.", "tab:verdict")

if "final_noise_ceiling.csv" in D:
    to_tex(D["final_noise_ceiling.csv"], "table4_noise_ceiling",
           "Next-day error against the estimated irreducible noise floor.",
           "tab:noise", bold_rows=(FINAL_ROW,))

if "final_per_city.csv" in D:
    to_tex(D["final_per_city.csv"], "table5_per_city",
           "Severe-day average precision for each held-out city.", "tab:percity",
           floatfmt="%.3f")

In [ ]:
# ── bundle everything ───────────────────────────────────────────────────────
zip_path = Path("dextra_paper_assets.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT.parent))
n_fig = len(list((OUT / "figures").glob("*.pdf")))
n_tab = len(list((OUT / "tables").glob("*.tex")))
print(f"\n{n_fig} figures (PNG + PDF) and {n_tab} LaTeX tables")
print(f"bundled into {zip_path} ({zip_path.stat().st_size/1e3:.0f} KB)")

if IN_COLAB:
    print("\nDownloading the zip to your computer...")
    colab_files.download(str(zip_path))
else:
    print(f"\nnot in Colab; the bundle is at {zip_path.resolve()}")

print("\nUse the .pdf figures in LaTeX - they are vector and stay sharp at any size.")